# Malicious URL Detection Model Tester

This notebook allows you to test the Random Forest model trained using lexical features as described in the paper:
**"Using Lexical Features for Malicious URL Detection - A Machine Learning Approach"**

In [ ]:
import pickle
import pandas as pd
import numpy as np
import os
import sys

# Add project root to path to allow importing src
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.analysis.lexical.features import extract_features
from src.analysis.lexical.ngram_features import TrigramFeatureExtractor

MODEL_PATH = "../models/lexical_rf.pkl"
TRIGRAM_PATH = "../models/trigram_extractor.pkl"

## 1. Load Model and Feature Extractor
We load the trained Random Forest model and the N-gram feature extractor.

In [ ]:
# Load the model artifacts
if not os.path.exists(MODEL_PATH) or not os.path.exists(TRIGRAM_PATH):
    print(f"❌ Model files not found. Please run the training script first.")
else:
    # Load Model
    with open(MODEL_PATH, "rb") as f:
        artifacts = pickle.load(f)
        model = artifacts["model"]
        feature_names = artifacts["feature_names"]
        print("✅ Random Forest Model loaded successfully.")
        
    # Load Trigram Extractor
    trigram_extractor = TrigramFeatureExtractor()
    trigram_extractor.load(TRIGRAM_PATH)
    print("✅ Trigram Extractor loaded successfully.")

## 2. Prediction Function
This function extracts features from a new URL and runs it through the model.

In [ ]:
def predict_url(url):
    """Extracts features and predicts if a URL is malicious."""
    try:
        # 1. Extract Lexical Features
        feats = extract_features(url)
        
        # 2. Extract N-gram Features
        trigram_feats = trigram_extractor.transform(url)
        feats.update(trigram_feats)
        
        # 3. Create DataFrame and align with training features
        feat_df = pd.DataFrame([feats])
        # Ensure all columns from training are present, fill missing with 0
        # This handles cases where new URLs don't hit certain trigrams or features
        feat_df = feat_df.reindex(columns=feature_names, fill_value=0)
        
        # 4. Predict
        prob = model.predict_proba(feat_df)[0][1]
        prediction = "Malicious" if prob > 0.5 else "Benign"
        
        # Visualization colors
        color = "\033[91m" if prediction == "Malicious" else "\033[92m"
        reset = "\033[0m"
        
        print(f"URL: {url}")
        print(f"Result: {color}{prediction}{reset} (Confidence: {prob:.2%})")
        
        print("-" * 30)
        return prob

    except Exception as e:
        print(f"Error processing {url}: {e}")
        return None

# Test some samples
print("Running test samples...")
predict_url("https://google.com")
predict_url("secure-login-account.top/verify")
predict_url("http://123.45.67.89/malware.exe")
predict_url("wikipedia.org")

## 3. Test Your Own URL
Enter a URL below to test it against the model.

In [ ]:
user_url = "https://gemini.google.com/"  # @param {type:"string"}
if user_url:
    predict_url(user_url)
else:
    print("Enter a URL in the field above (or edit the code) to test it.")